# Keyword cannibalization checker (Search Console API + Sheets report)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lazarinastoy/mlforseo-code/blob/main/keyword-cannibalization-checker/keyword_cannibalization_checker.ipynb)

A free notebook by **Lazarina Stoy, MLforSEO**.

"Keyword cannibalization" gets blamed for a lot, but most queries where two of your pages show up are perfectly healthy: brand searches with sitelinks, a product page and a guide answering different intents, or one strong page with a weak page trailing on page two. This notebook finds the cases that are worth your time.

**What it does**

1. Pulls **query + page** data (optionally by **date**) from the Search Console API, with pagination. Or reads a CSV export.
2. Cleans URL variants that are really one page (`#fragments`, tracking parameters, missing trailing slashes).
3. Finds queries where **two or more pages** get a meaningful share of impressions or clicks.
4. Quantifies the split: click and impression shares, position spread, and (with dates) how often the ranking page **rotates**.
5. Scores **severity** and separates likely-legitimate cases (brand + sitelinks, different page types and intents, one clear winner) from true cannibalisation candidates.
6. Suggests an action: **consolidate**, **differentiate**, **internal-link** or **leave**.
7. Exports a Sheets-ready **.xlsx** report with conditional formatting, plus CSVs.

**What you need:** nothing for the sample run. For your own data, either a Search Console export with page and query columns, or API access (set-up in step 1). The Search Console API is free.

**Sample data:** `sample_data/gsc_sample_SYNTHETIC.csv` is **synthetic**. "Brewly" is a made-up coffee shop and every number is simulated (the generator script is in the same folder).

In [1]:
# Colab already has these; locally: pip install -r requirements.txt
import os, re, math
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
import pandas as pd

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 160)

## 1. Settings and data source

Pick one of three sources:

* `'sample'`: the synthetic Brewly export. No account needed.
* `'csv'`: your own export with the columns `query`, `page`, `clicks`, `impressions`, `position` (and `date` if you have it). Exports from the Search Console API, the Search Analytics for Sheets add-on and the BigQuery bulk export all work; common column names are mapped for you. The web interface's export doesn't combine query and page, so it won't work here.
* `'api'`: pull directly from the Search Console API (see the authentication notes in the next cell).

**Include the date dimension** (`USE_DATE = True`) if you can. It makes the export bigger, but it's the only way to see rotation: the same query swapping between your pages from one day to the next, which is the clearest sign of real cannibalisation.

In [2]:
SOURCE = 'sample'            # 'sample', 'csv' or 'api'
CSV_PATH = 'gsc_export.csv'  # used when SOURCE = 'csv'

# API settings (used when SOURCE = 'api')
SITE_URL = 'sc-domain:example.com'   # or the exact URL-prefix property, e.g. 'https://www.example.com/'
START_DATE = '2026-07-01'
END_DATE = '2026-09-30'              # Search Console keeps 16 months; the last 2-3 days are incomplete
USE_DATE = True                      # adds the date dimension for rotation analysis
SEARCH_TYPE = 'web'
MAX_ROWS = 200_000                   # stop paging after this many rows
PAGE_FILTER_REGEX = ''               # optional, e.g. r'/blog/' to audit one section
QUERY_FILTER_REGEX = ''              # optional, e.g. r'grinder|kettle'
AUTH_METHOD = 'service_account'      # 'service_account' or 'oauth' (see below)
SERVICE_ACCOUNT_FILE = 'service-account.json'
OAUTH_CLIENT_FILE = 'client_secret.json'

# Your brand terms: queries containing them get the "brand" treatment
BRAND_TERMS = ['brewly']

### Authentication for `SOURCE = 'api'` (skip for the sample)

The Search Console API needs the read-only scope `https://www.googleapis.com/auth/webmasters.readonly`. Two ways to get it:

**Option A: service account (simplest, works in Colab and locally).**
1. In Google Cloud, create a project, enable the **Google Search Console API**, then create a **service account** and download its JSON key.
2. In Search Console, open **Settings > Users and permissions** and add the service account's email address as a **Restricted** user on the property.
3. Upload the JSON key to Colab (folder icon on the left) and set `SERVICE_ACCOUNT_FILE`. Treat this file like a password: don't commit it or share the notebook with it attached.

**Option B: your own Google account (OAuth).**
1. In the same Cloud project, create an **OAuth client ID** of type **Desktop app** and download `client_secret.json`. On the OAuth consent screen, add yourself as a test user.
2. Locally, the cell below opens a browser window for you to sign in.
3. In Colab, upload `client_secret.json` and run the `gcloud` command shown in the cell's message, then re-run the cell. The cell reads the resulting credentials with `google.auth.default()`.

Why not just `google.colab.auth.authenticate_user()`? It's the one-click Colab sign-in, and the cell tries it first, but it signs you in with Cloud scopes that don't include Search Console, so on its own it usually ends in an "insufficient authentication scopes" error. That's why Options A and B exist.

In [3]:
SCOPES = ['https://www.googleapis.com/auth/webmasters.readonly']

def get_credentials():
    """Return Google credentials with the Search Console read-only scope."""
    if AUTH_METHOD == 'service_account':
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

    import google.auth
    try:
        from google.colab import auth as colab_auth   # only exists inside Colab
        in_colab = True
    except ImportError:
        in_colab = False

    if in_colab:
        # Try Colab's own sign-in first, then credentials created with gcloud (see the message below)
        colab_auth.authenticate_user()
        creds, _ = google.auth.default(scopes=SCOPES)
        print('Using Colab / application-default credentials. If the API answers "insufficient '
              'authentication scopes", run this in a cell, sign in, then re-run this cell:\n'
              f'!gcloud auth application-default login --no-launch-browser --client-id-file={OAUTH_CLIENT_FILE} '
              '--scopes=https://www.googleapis.com/auth/webmasters.readonly,https://www.googleapis.com/auth/cloud-platform')
        return creds

    from google_auth_oauthlib.flow import InstalledAppFlow
    flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_FILE, SCOPES)
    return flow.run_local_server(port=0)


def fetch_search_console(site_url, start_date, end_date, use_date=True, search_type='web',
                         max_rows=200_000, page_regex='', query_regex=''):
    """Page through searchanalytics.query (25,000 rows per request) and return a DataFrame."""
    from googleapiclient.discovery import build
    service = build('searchconsole', 'v1', credentials=get_credentials(), cache_discovery=False)

    dimensions = (['date'] if use_date else []) + ['query', 'page']
    filters = []
    if page_regex:
        filters.append({'dimension': 'page', 'operator': 'includingRegex', 'expression': page_regex})
    if query_regex:
        filters.append({'dimension': 'query', 'operator': 'includingRegex', 'expression': query_regex})

    rows, start_row, row_limit = [], 0, 25_000      # 25,000 is the API maximum per request
    while start_row < max_rows:
        body = {'startDate': start_date, 'endDate': end_date, 'dimensions': dimensions,
                'type': search_type, 'rowLimit': row_limit, 'startRow': start_row, 'dataState': 'final'}
        if filters:
            body['dimensionFilterGroups'] = [{'groupType': 'and', 'filters': filters}]
        resp = service.searchanalytics().query(siteUrl=site_url, body=body).execute()
        batch = resp.get('rows', [])
        for r in batch:
            rows.append(dict(zip(dimensions, r['keys']), clicks=r['clicks'], impressions=r['impressions'],
                             ctr=r['ctr'], position=r['position']))
        print(f'  fetched {len(batch):,} rows from row {start_row:,}')
        if len(batch) < row_limit:
            break
        start_row += row_limit
    return pd.DataFrame(rows)

### Load the data

Column names from the common export tools are mapped to `date`, `query`, `page`, `clicks`, `impressions`, `position`. If yours differ, rename them in `COLUMN_ALIASES`.

In [4]:
COLUMN_ALIASES = {
    'query': ['query', 'queries', 'top queries', 'search query', 'keyword'],
    'page': ['page', 'url', 'landing page', 'address', 'top pages'],
    'date': ['date', 'day', 'data_date'],
    'clicks': ['clicks', 'click', 'url_clicks'],
    'impressions': ['impressions', 'impression', 'impr'],
    'position': ['position', 'avg. position', 'average position', 'avg position', 'pos'],
}

def standardise_columns(df):
    lookup = {alias: std for std, aliases in COLUMN_ALIASES.items() for alias in aliases}
    df = df.rename(columns={c: lookup.get(c.strip().lower(), c) for c in df.columns})
    missing = {'query', 'page', 'clicks', 'impressions', 'position'} - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {missing}. Found: {list(df.columns)}')
    for col in ['clicks', 'impressions', 'position']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    return df


SAMPLE_PATH = 'sample_data/gsc_sample_SYNTHETIC.csv'
SAMPLE_URL = 'https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/keyword-cannibalization-checker/sample_data/gsc_sample_SYNTHETIC.csv'

if SOURCE == 'sample':
    raw = pd.read_csv(SAMPLE_PATH if os.path.exists(SAMPLE_PATH) else SAMPLE_URL)
    print('Using the SYNTHETIC sample (made-up site "Brewly", simulated numbers).')
elif SOURCE == 'csv':
    raw = pd.read_csv(CSV_PATH)
elif SOURCE == 'api':
    raw = fetch_search_console(SITE_URL, START_DATE, END_DATE, USE_DATE, SEARCH_TYPE,
                               MAX_ROWS, PAGE_FILTER_REGEX, QUERY_FILTER_REGEX)
else:
    raise ValueError("SOURCE must be 'sample', 'csv' or 'api'")

gsc = standardise_columns(raw)
HAS_DATE = 'date' in gsc.columns
print(f'{len(gsc):,} rows, {gsc["query"].nunique():,} queries, {gsc["page"].nunique():,} URLs, date dimension: {HAS_DATE}')
gsc.head()

Using the SYNTHETIC sample (made-up site "Brewly", simulated numbers).
1,864 rows, 20 queries, 25 URLs, date dimension: True


,date,query,page,clicks,impressions,ctr,position
0,2026-08-01,aeropress recipe,https://www.brewly.example/recipes/aeropress/,24,140,0.1714,2.8
1,2026-08-01,best coffee grinder,https://www.brewly.example/blog/best-burr-grinders-2025/,0,86,0.0000,16.7
2,2026-08-01,best coffee grinder,https://www.brewly.example/blog/best-coffee-grinders/,17,226,0.0752,5.4
3,2026-08-01,best coffee grinder,https://www.brewly.example/collections/grinders/,0,19,0.0000,20.5
4,2026-08-01,best coffee grinders,https://www.brewly.example/blog/best-coffee-grinders/,3,95,0.0316,7.6


## 2. Merge URL variants that are really one page

Search Console reports jump links (`/guide/#ratio`), tracking parameters and slash variants as separate pages. Left alone, they look like two pages competing for one query. This step strips fragments and tracking parameters, lower-cases the host and adds a trailing slash, then re-aggregates.

Average position is re-weighted by impressions, which is how Search Console itself averages position.

In [5]:
TRACKING_PARAMS = re.compile(r'^(utm_\w+|gclid|fbclid|msclkid|mc_cid|mc_eid|ref|srsltid)$', re.I)

def normalise_url(url):
    parts = urlsplit(str(url).strip())
    query = urlencode([(k, v) for k, v in parse_qsl(parts.query) if not TRACKING_PARAMS.match(k)])
    path = parts.path or '/'
    if not path.endswith('/') and '.' not in path.rsplit('/', 1)[-1]:   # leave /file.pdf alone
        path += '/'
    return urlunsplit((parts.scheme.lower() or 'https', parts.netloc.lower(), path, query, ''))


gsc['page_raw'] = gsc['page']
gsc['page'] = gsc['page'].map(normalise_url)
merged = gsc.groupby('page')['page_raw'].nunique()
print(f'{gsc["page_raw"].nunique():,} raw URLs -> {gsc["page"].nunique():,} pages after normalising')
print('Pages that had variants:', list(merged[merged > 1].index))

keys = (['date'] if HAS_DATE else []) + ['query', 'page']
gsc['pos_x_impr'] = gsc['position'] * gsc['impressions']
gsc = (gsc.groupby(keys, as_index=False)[['clicks', 'impressions', 'pos_x_impr']].sum())
gsc['position'] = (gsc['pos_x_impr'] / gsc['impressions'].where(gsc['impressions'] > 0)).round(1)
gsc = gsc.drop(columns='pos_x_impr')

25 raw URLs -> 22 pages after normalising
Pages that had variants: ['https://www.brewly.example/blog/chemex-vs-v60/', 'https://www.brewly.example/blog/cold-brew-guide/', 'https://www.brewly.example/recipes/aeropress/']


## 3. Page types (reused from the intent mismatch audit)

The same idea as the companion [intent mismatch audit](../intent-mismatch-audit/): URL patterns map each page to a type. When two pages of **different** types share a query, Google is often serving a mixed-intent results page on purpose (a guide for people learning, a product page for people buying). When two pages of the **same** type share it, that's the classic cannibalisation pattern.

Edit the patterns to fit your site. The first match wins.

In [6]:
PAGE_TYPES = [
    ('home',     r'^/$'),
    ('product',  r'^/(products?|shop|collections?|category|categories|features|pricing|plans|services|solutions)(/|$)'),
    ('company',  r'^/(about|team|careers|contact|press|promotions?)(/|$)'),
    ('resource', r'^/(blog|guides?|learn|resources|recipes|glossary|academy|news)(/|$)'),
]

def page_type(url):
    path = urlsplit(url).path
    for name, rx in PAGE_TYPES:
        if re.search(rx, path):
            return name
    return 'other'

page_types = {p: page_type(p) for p in gsc['page'].unique()}
pd.Series(page_types).value_counts()

resource    14
product      4
company      3
home         1
Name: count, dtype: int64

## 4. Find queries where several pages compete

A page counts as **competing** for a query when it gets at least `MIN_IMPR_SHARE` of the query's impressions **and** at least `MIN_IMPRESSIONS` impressions (or any clicks). This floor stops a page that once appeared at position 40 from being reported as a rival.

`MIN_QUERY_IMPRESSIONS` skips queries too small to matter. Raise it on large sites.

In [7]:
MIN_QUERY_IMPRESSIONS = 100   # ignore queries with fewer impressions in the period
MIN_IMPR_SHARE = 0.10         # a page needs at least 10% of the query's impressions...
MIN_IMPRESSIONS = 20          # ...and at least 20 impressions (or 1+ click) to count as competing

qp = gsc.groupby(['query', 'page'], as_index=False).agg(clicks=('clicks', 'sum'), impressions=('impressions', 'sum'))
# impression-weighted average position per query and page
weighted = gsc.assign(w=gsc['position'] * gsc['impressions']).groupby(['query', 'page'])[['w', 'impressions']].sum()
pos = (weighted['w'] / weighted['impressions'].clip(lower=1)).round(1).rename('position')
qp = qp.merge(pos.reset_index(), on=['query', 'page'])
if HAS_DATE:
    qp = qp.merge(gsc.groupby(['query', 'page'])['date'].nunique().rename('days_shown').reset_index(), on=['query', 'page'])

totals = qp.groupby('query')[['clicks', 'impressions']].sum().add_prefix('query_')
qp = qp.join(totals, on='query')
qp['click_share'] = (qp['clicks'] / qp['query_clicks'].where(qp['query_clicks'] > 0)).fillna(0).round(3)
qp['impr_share'] = (qp['impressions'] / qp['query_impressions']).round(3)
qp['page_type'] = qp['page'].map(page_types)
qp['competing'] = ((qp['query_impressions'] >= MIN_QUERY_IMPRESSIONS)
                   & (qp['impr_share'] >= MIN_IMPR_SHARE)
                   & ((qp['impressions'] >= MIN_IMPRESSIONS) | (qp['clicks'] > 0)))

competing = qp[qp['competing']].copy()
n_pages = competing.groupby('query')['page'].nunique()
candidates = n_pages[n_pages >= 2].index
detail = (competing[competing['query'].isin(candidates)]
          .sort_values(['query', 'clicks', 'impressions'], ascending=[True, False, False]))
print(f'{qp["query"].nunique():,} queries checked, {len(candidates)} with 2+ competing pages')
detail[['query', 'page', 'page_type', 'clicks', 'impressions', 'click_share', 'impr_share', 'position']]

20 queries checked, 12 with 2+ competing pages


,query,page,page_type,clicks,impressions,click_share,impr_share,position
2,best coffee grinder,https://www.brewly.example/blog/best-coffee-grinders/,resource,421,10017,0.639,0.487,7.9
1,best coffee grinder,https://www.brewly.example/blog/best-burr-grinders-2025/,resource,223,7598,0.338,0.369,9.3
3,best coffee grinder,https://www.brewly.example/collections/grinders/,product,15,2966,0.023,0.144,17.0
5,best coffee grinders,https://www.brewly.example/blog/best-coffee-grinders/,resource,174,4466,0.596,0.483,8.2
4,best coffee grinders,https://www.brewly.example/blog/best-burr-grinders-2025/,resource,109,3474,0.373,0.376,9.2
6,best coffee grinders,https://www.brewly.example/collections/grinders/,product,9,1308,0.031,0.141,17.8
7,brewly,https://www.brewly.example/,home,3238,10004,0.350,0.250,1.0
10,brewly,https://www.brewly.example/shop/,product,2350,10004,0.254,0.250,2.0
8,brewly,https://www.brewly.example/about/,company,1946,10004,0.210,0.250,2.5
9,brewly,https://www.brewly.example/contact/,company,1712,10004,0.185,0.250,3.0


## 5. Rotation over time (needs the date dimension)

For each query and day, the **leader** is the competing page with the best average position that day. A **leader change** is a day when a different page leads than on the previous day it had data. Lots of changes means Google can't settle on which of your pages answers the query, which is the strongest signal of real cannibalisation.

* `rotation_rate`: leader changes divided by the number of days with data (0 = one page always leads).
* `days_multi_share`: share of days when two or more competing pages appeared at the same time. High values with low rotation usually mean Google is *choosing* to show both (common for mixed intent).

In [8]:
def rotation_stats(daily):
    """daily: rows for one query (competing pages only) with date, page, position."""
    by_day = daily.sort_values(['date', 'position']).groupby('date')
    leaders = by_day['page'].first()
    changes = int((leaders != leaders.shift()).sum() - 1)
    return pd.Series({
        'days_with_data': len(leaders),
        'leader_changes': max(changes, 0),
        'rotation_rate': round(max(changes, 0) / max(len(leaders) - 1, 1), 3),
        'days_multi_share': round((by_day['page'].nunique() >= 2).mean(), 3),
    })

if HAS_DATE:
    daily = gsc.merge(detail[['query', 'page']], on=['query', 'page'])
    daily = daily[daily['impressions'] > 0]
    rotation = pd.DataFrame({q: rotation_stats(g) for q, g in daily.groupby('query')}).T
    # Day-by-day leader table for the report
    daily_leader = (daily.sort_values(['query', 'date', 'position']).groupby(['query', 'date'])
                    .agg(leader=('page', 'first'), leader_position=('position', 'first'),
                         pages_shown=('page', 'nunique')).reset_index())
else:
    rotation = pd.DataFrame(index=candidates, columns=['days_with_data', 'leader_changes', 'rotation_rate', 'days_multi_share'])
    daily_leader = pd.DataFrame()
rotation.sort_values('rotation_rate', ascending=False)

,days_with_data,leader_changes,rotation_rate,days_multi_share
best coffee grinder,56.0,9.0,0.164,0.929
best coffee grinders,56.0,9.0,0.164,0.911
french press vs pour over,56.0,9.0,0.164,0.518
pour over vs french press,56.0,9.0,0.164,0.589
cold brew coffee ratio,56.0,6.0,0.109,0.893
cold brew ratio,56.0,6.0,0.109,0.821
espresso machine cleaning,56.0,2.0,0.036,0.857
brewly,56.0,0.0,0.000,1.000
brewly grinder,56.0,0.0,0.000,1.000
burr grinder,56.0,0.0,0.000,1.000


## 6. Severity score, legitimate cases and suggested action

**Severity (0-100)** adds four parts. Each is capped so no single signal dominates:

| Part | Max points | What it measures |
|---|---|---|
| Split | 40 | How evenly clicks (or impressions, if there are no clicks) are shared. A 50/50 split scores full points; 90/10 scores little. |
| Rotation | 30 | `rotation_rate` (0.3 or more scores full points). Without dates, this part is 0, so scores run lower. |
| Proximity | 10 | The top two pages rank within 2 positions of each other (full points) to 10 apart (none). Pages far apart are rarely fighting. |
| Demand | 20 | The query's impressions, on a log scale relative to your biggest candidate query, so big queries rise to the top. |

Bands: **high** 60+, **medium** 35-59, **low** under 35. The table is sorted by action (consolidate first, leave last), then severity, so legitimate cases with big numbers, like brand queries, sink to the bottom.

**Likely legitimate** (checked in this order):

* **brand / sitelinks**: the query contains a brand term and the home page or a company page is involved, or every competing page ranks in the top 3 (Google grouping your results).
* **one clear winner**: the top page has 80%+ of clicks (impressions if there are no clicks).
* **cannibalisation candidate**: two or more competing pages of the **same** type (for example two blog posts), even if a page of another type is also in the mix.
* **different page types**: the competing pages are all of different types (for example a product page and a guide). Usually a mixed-intent results page. Check the SERP before changing anything.

**Suggested action:**

* **leave**: brand / sitelinks, or low severity with a clear winner.
* **internal-link**: one clear winner, or different page types that Google shows side by side without rotating (or at low or medium severity). Link from the weaker page to the stronger one with the query as anchor text, so each page has a clear job.
* **differentiate**: different page types at high severity that also rotate, or a same-type medium case. Sharpen each page's angle and title so they answer different needs, and cross-link them.
* **consolidate**: same page type at high severity. Merge the content into the stronger page and 301-redirect the other. The suggested page to keep is the same-type page with the most clicks for this query (ties go to the page with more clicks site-wide).

These are heuristics, so read them as a shortlist to check by hand, with the live SERP open.

In [9]:
BRAND_RX = re.compile(r'\b(?:' + '|'.join(map(re.escape, BRAND_TERMS)) + r')\b', re.I) if BRAND_TERMS else None
page_clicks_sitewide = gsc.groupby('page')['clicks'].sum()
max_impr = detail.groupby('query')['impressions'].sum().max() if len(detail) else 1

def assess(q, group):
    g = group.copy()
    weight = 'clicks' if g['clicks'].sum() > 0 else 'impressions'
    g['rank_key'] = list(zip(g[weight], g['impressions'], g['page'].map(page_clicks_sitewide)))
    g = g.sort_values('rank_key', ascending=False)
    shares = g[weight] / g[weight].sum()
    top_share = shares.iloc[0]
    by_pos = g.sort_values('position')
    spread = round(g['position'].max() - g['position'].min(), 1)
    gap_top2 = abs(by_pos['position'].iloc[1] - by_pos['position'].iloc[0])
    rot = rotation.loc[q] if q in rotation.index else None
    rotation_rate = float(rot['rotation_rate']) if rot is not None and pd.notna(rot['rotation_rate']) else 0.0

    # Severity parts
    n = len(g)
    evenness = (1 - top_share) / (1 - 1 / n) if n > 1 else 0       # 1 = perfectly even split
    split_pts = 40 * min(evenness / 0.8, 1)
    rotation_pts = 30 * min(rotation_rate / 0.3, 1)
    proximity_pts = 10 * min(max((10 - gap_top2) / 8, 0), 1)
    demand_pts = 20 * math.log1p(g['impressions'].sum()) / math.log1p(max_impr)
    severity = round(split_pts + rotation_pts + proximity_pts + demand_pts)
    band = 'high' if severity >= 60 else 'medium' if severity >= 35 else 'low'

    types = set(g['page_type'])
    type_counts = g['page_type'].value_counts()
    same_type = type_counts[type_counts >= 2].index.tolist()     # types with 2+ competing pages
    is_brand = bool(BRAND_RX and BRAND_RX.search(q))
    stable_together = HAS_DATE and rotation_rate < 0.05 and rot is not None and float(rot['days_multi_share']) >= 0.7
    if (is_brand and types & {'home', 'company', 'product'}) or g['position'].max() <= 3:
        case, action = 'brand / sitelinks', 'leave'
    elif top_share >= 0.8:
        case = 'one clear winner'
        action = 'leave' if band == 'low' else 'internal-link'
    elif same_type:
        # Two or more pages of the SAME type share the query: the classic pattern,
        # even if a page of another type (e.g. a category page) is also in the mix
        case = 'cannibalisation candidate' + (' (+ other page type)' if len(types) > 1 else '')
        action = {'high': 'consolidate', 'medium': 'differentiate', 'low': 'internal-link'}[band]
        keep = g[g['page_type'].isin(same_type)]['page'].iloc[0]
        g = pd.concat([g[g['page'] == keep], g[g['page'] != keep]])
    else:
        case = 'different page types (likely mixed intent)'
        action = 'differentiate' if (band == 'high' and not stable_together) else 'internal-link'

    return pd.Series({
        'severity': severity, 'band': band, 'case': case, 'action': action,
        'pages': n, 'keep_page': g['page'].iloc[0],
        'other_pages': ' | '.join(g['page'].iloc[1:]),
        'page_types': ', '.join(sorted(types)),
        'clicks': int(g['clicks'].sum()), 'impressions': int(g['impressions'].sum()),
        'top_share': round(top_share, 3), 'share_basis': weight,
        'best_position': g['position'].min(), 'position_spread': spread,
        'rotation_rate': rotation_rate if HAS_DATE else None,
        'leader_changes': int(rot['leader_changes']) if HAS_DATE and rot is not None else None,
        'days_multi_share': float(rot['days_multi_share']) if HAS_DATE and rot is not None else None,
    })

ACTION_ORDER = {'consolidate': 0, 'differentiate': 1, 'internal-link': 2, 'leave': 3}
report = pd.DataFrame([assess(q, g) for q, g in detail.groupby('query')], index=sorted(detail['query'].unique()))
report = (report.rename_axis('query').reset_index()
          .assign(_order=lambda d: d['action'].map(ACTION_ORDER))
          .sort_values(['_order', 'severity'], ascending=[True, False]).drop(columns='_order').reset_index(drop=True))
report[['query', 'severity', 'band', 'case', 'action', 'pages', 'top_share', 'position_spread',
        'rotation_rate', 'keep_page']]

,query,severity,band,case,action,pages,top_share,position_spread,rotation_rate,keep_page
0,pour over vs french press,82,high,cannibalisation candidate,consolidate,2,0.567,1.0,0.164,https://www.brewly.example/blog/french-press-vs-pour-over/
1,french press vs pour over,81,high,cannibalisation candidate,consolidate,2,0.624,1.0,0.164,https://www.brewly.example/blog/french-press-vs-pour-over/
2,cold brew ratio,80,high,cannibalisation candidate,consolidate,2,0.572,0.1,0.109,https://www.brewly.example/recipes/cold-brew/
3,cold brew coffee ratio,78,high,cannibalisation candidate,consolidate,2,0.510,0.3,0.109,https://www.brewly.example/recipes/cold-brew/
4,best coffee grinders,74,high,cannibalisation candidate (+ other page type),consolidate,3,0.596,9.6,0.164,https://www.brewly.example/blog/best-coffee-grinders/
5,best coffee grinder,72,high,cannibalisation candidate (+ other page type),consolidate,3,0.639,9.1,0.164,https://www.brewly.example/blog/best-coffee-grinders/
6,espresso machine cleaning,58,medium,cannibalisation candidate,differentiate,2,0.725,2.4,0.036,https://www.brewly.example/blog/espresso-machine-mainten...
7,burr grinder,56,medium,different page types (likely mixed intent),internal-link,2,0.725,3.1,0.000,https://www.brewly.example/blog/what-is-a-burr-grinder/
8,brewly,70,high,brand / sitelinks,leave,4,0.350,2.0,0.000,https://www.brewly.example/
9,brewly grinder,65,high,brand / sitelinks,leave,2,0.607,1.3,0.000,https://www.brewly.example/products/burr-grinder-pro/


### Reading the sample result

In the synthetic Brewly data you should see:

* **"french press vs pour over"**, **"best coffee grinder"** and **"cold brew ratio"** at the top as *consolidate*: two similar pages taking turns. The grinder case also includes a category page, but the real problem is the two list posts: this year's and last year's, the typical "we wrote it again instead of updating it" case.
* **"espresso machine cleaning"**: two maintenance guides splitting the query without much rotation. Medium severity, so the suggestion is to *differentiate* them (or merge, if they really say the same thing).
* **"burr grinder"**: a product page and an explainer shown together every day with no rotation. Flagged as *different page types*, with an *internal-link* suggestion: probably a mixed-intent SERP where both pages have a job.
* **"brewly"** and **"brewly grinder"**: brand queries with sitelinks. Leave them.
* **"how to descale an espresso machine"**: one page wins almost everything. At most, add an internal link.
* **"aeropress recipe"**, **"chemex vs v60"** and **"pour over ratio"** don't appear at all: their second URL was a tracking-parameter or slash variant (merged in step 2) or sat below the impression-share floor.

## 7. Page pairs that keep colliding

Borrowed from the **OverlapSummary** tab of the free On-page Duplication Analysis sheet: for every pair of pages that compete on at least one query, how many queries they share and how much demand sits on those queries. A pair that shares many queries is a stronger merge or differentiation candidate than a pair that collides once.

In [10]:
from itertools import combinations

pairs = []
for q, g in detail.groupby('query'):
    for a, b in combinations(sorted(g['page']), 2):
        pairs.append({'page_a': a, 'page_b': b, 'query': q, 'impressions': int(g['impressions'].sum())})
pairs = pd.DataFrame(pairs)
queries_per_page = qp[qp['impressions'] >= MIN_IMPRESSIONS].groupby('page')['query'].apply(set)

if len(pairs):
    page_pairs = (pairs.groupby(['page_a', 'page_b'])
                  .agg(shared_queries=('query', 'nunique'), shared_query_impressions=('impressions', 'sum'),
                       queries=('query', lambda s: ' | '.join(sorted(s)))).reset_index())
    def jaccard(r):
        a, b = queries_per_page.get(r['page_a'], set()), queries_per_page.get(r['page_b'], set())
        return round(len(a & b) / len(a | b), 2) if a | b else 0
    page_pairs['query_overlap'] = page_pairs.apply(jaccard, axis=1)
    page_pairs = page_pairs.sort_values(['shared_queries', 'shared_query_impressions'], ascending=False)
else:
    page_pairs = pd.DataFrame(columns=['page_a', 'page_b', 'shared_queries', 'shared_query_impressions', 'queries', 'query_overlap'])
page_pairs

,page_a,page_b,shared_queries,shared_query_impressions,queries,query_overlap
10,https://www.brewly.example/blog/espresso-machine-mainten...,https://www.brewly.example/blog/how-to-descale-an-espres...,3,27875,espresso machine cleaning | how to descale an espresso m...,1.00
6,https://www.brewly.example/blog/best-burr-grinders-2025/,https://www.brewly.example/blog/best-coffee-grinders/,2,29829,best coffee grinder | best coffee grinders,1.00
7,https://www.brewly.example/blog/best-burr-grinders-2025/,https://www.brewly.example/collections/grinders/,2,29829,best coffee grinder | best coffee grinders,1.00
8,https://www.brewly.example/blog/best-coffee-grinders/,https://www.brewly.example/collections/grinders/,2,29829,best coffee grinder | best coffee grinders,1.00
9,https://www.brewly.example/blog/cold-brew-guide/,https://www.brewly.example/recipes/cold-brew/,2,27166,cold brew coffee ratio | cold brew ratio,0.67
11,https://www.brewly.example/blog/french-press-vs-pour-over/,https://www.brewly.example/blog/pour-over-coffee-guide/,2,12973,french press vs pour over | pour over vs french press,0.67
0,https://www.brewly.example/,https://www.brewly.example/about/,1,40016,brewly,0.50
1,https://www.brewly.example/,https://www.brewly.example/contact/,1,40016,brewly,0.50
3,https://www.brewly.example/,https://www.brewly.example/shop/,1,40016,brewly,0.50
4,https://www.brewly.example/about/,https://www.brewly.example/contact/,1,40016,brewly,1.00


## 8. Export: Sheets-ready report and CSVs

`cannibalization_report.xlsx` has these tabs (upload it to Google Drive and open it with Google Sheets, or use **File > Import** in an existing sheet):

* **How to read**: the method and thresholds used for this run.
* **Summary**: counts by band and action (live `COUNTIF` formulas, so they update if you edit actions by hand).
* **Cannibalisation**: one row per query, colour-coded severity and action, an empty **Decision** column for your call, and filters.
* **Query x page detail**: every competing page with its shares and position.
* **Page pairs**: pairs of pages that share queries.
* **Daily leader**: which page led each query each day (only with the date dimension).

The same tables are saved as CSVs.

In [11]:
from openpyxl import Workbook
from openpyxl.formatting.rule import CellIsRule, ColorScaleRule, DataBarRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

HEADER_FILL = PatternFill('solid', fgColor='251046')
HEADER_FONT = Font(bold=True, color='FFFFFF')
ACTION_FILLS = {'consolidate': 'F4C7C3', 'differentiate': 'FCE8B2', 'internal-link': 'C9DAF8', 'leave': 'D9EAD3'}

def write_table(ws, df, widths=None):
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append([None if (isinstance(v, float) and math.isnan(v)) else v for v in row])
    for cell in ws[1]:
        cell.fill, cell.font = HEADER_FILL, HEADER_FONT
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    ws.freeze_panes = 'B2'
    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(df.columns, start=1):
        width = (widths or {}).get(col) or min(max(len(str(col)), *(len(str(v)) for v in df[col].head(200)), 8) + 2, 60)
        ws.column_dimensions[get_column_letter(i)].width = width

def col(df, name):
    return get_column_letter(list(df.columns).index(name) + 1)

def build_workbook(path):
    wb = Workbook()
    ws = wb.active
    ws.title = 'How to read'
    notes = [
        ['Keyword cannibalization checker', 'by Lazarina Stoy, MLforSEO (mlforseo.com)'],
        ['Data source', 'SYNTHETIC sample (made-up site)' if SOURCE == 'sample' else SOURCE],
        ['Date dimension', 'yes' if HAS_DATE else 'no (rotation not measured)'],
        ['Thresholds', f'query impressions >= {MIN_QUERY_IMPRESSIONS}; page needs >= {MIN_IMPR_SHARE:.0%} of impressions and >= {MIN_IMPRESSIONS} impressions or a click'],
        ['Severity', 'split (40) + rotation (30) + proximity of top two positions (10) + demand (20). high >= 60, medium 35-59, low < 35'],
        ['consolidate', 'Same page type, high severity: merge into keep_page and 301 the others'],
        ['differentiate', 'Give each page a distinct angle, title and intent; cross-link them'],
        ['internal-link', 'Link the weaker page(s) to keep_page with the query as anchor text'],
        ['leave', 'Brand/sitelinks or a clear winner at low severity: no action needed'],
        ['Caveat', 'Heuristics, not verdicts. Check the live SERP and the pages before merging anything.'],
    ]
    for r in notes:
        ws.append(r)
    ws['A1'].font = Font(bold=True, size=14)
    ws.column_dimensions['A'].width, ws.column_dimensions['B'].width = 18, 110

    rep = report.copy()
    rep.insert(rep.columns.get_loc('action') + 1, 'decision', '')
    ws_rep = wb.create_sheet('Cannibalisation')
    write_table(ws_rep, rep, widths={'query': 34, 'keep_page': 50, 'other_pages': 70, 'case': 34, 'decision': 18})
    last = ws_rep.max_row

    summary = wb.create_sheet('Summary', 1)
    summary.append(['Group', 'Value', 'Queries'])
    band_col, action_col = col(rep, 'band'), col(rep, 'action')
    for value in ['high', 'medium', 'low']:
        summary.append(['band', value, f"=COUNTIF(Cannibalisation!{band_col}2:{band_col}{last},\"{value}\")"])
    for value in ['consolidate', 'differentiate', 'internal-link', 'leave']:
        summary.append(['action', value, f"=COUNTIF(Cannibalisation!{action_col}2:{action_col}{last},\"{value}\")"])
    summary.append(['total', 'queries with 2+ competing pages', f'=COUNTA(Cannibalisation!A2:A{last})'])
    for cell in summary[1]:
        cell.fill, cell.font = HEADER_FILL, HEADER_FONT
    summary.column_dimensions['A'].width, summary.column_dimensions['B'].width = 10, 34

    if last > 1:
        sev = col(rep, 'severity')
        ws_rep.conditional_formatting.add(f'{sev}2:{sev}{last}', ColorScaleRule(
            start_type='num', start_value=0, start_color='FFFFFF',
            mid_type='num', mid_value=45, mid_color='FCE8B2', end_type='num', end_value=90, end_color='E06666'))
        act = col(rep, 'action')
        for action, colour in ACTION_FILLS.items():
            ws_rep.conditional_formatting.add(f'{act}2:{act}{last}', CellIsRule(
                operator='equal', formula=[f'"{action}"'], fill=PatternFill('solid', fgColor=colour)))
        share = col(rep, 'top_share')
        ws_rep.conditional_formatting.add(f'{share}2:{share}{last}', DataBarRule(
            start_type='num', start_value=0, end_type='num', end_value=1, color='8448EF'))

    det = detail[['query', 'page', 'page_type', 'clicks', 'impressions', 'click_share', 'impr_share', 'position']
                 + (['days_shown'] if HAS_DATE else [])]
    ws_det = wb.create_sheet('Query x page detail')
    write_table(ws_det, det, widths={'query': 34, 'page': 55})
    if ws_det.max_row > 1:
        for name in ['click_share', 'impr_share']:
            c = col(det, name)
            ws_det.conditional_formatting.add(f'{c}2:{c}{ws_det.max_row}', DataBarRule(
                start_type='num', start_value=0, end_type='num', end_value=1, color='8448EF'))

    write_table(wb.create_sheet('Page pairs'), page_pairs, widths={'page_a': 50, 'page_b': 50, 'queries': 60})
    if HAS_DATE and len(daily_leader):
        dl = daily_leader.assign(date=daily_leader['date'].dt.strftime('%Y-%m-%d'))
        write_table(wb.create_sheet('Daily leader'), dl, widths={'query': 34, 'leader': 55})
    wb.save(path)

os.makedirs('output', exist_ok=True)
build_workbook('output/cannibalization_report.xlsx')
report.to_csv('output/cannibalization_report.csv', index=False)
detail.to_csv('output/query_page_detail.csv', index=False)
page_pairs.to_csv('output/page_pairs.csv', index=False)
if HAS_DATE:
    daily_leader.to_csv('output/daily_leader.csv', index=False)
print('Saved to output/:', sorted(os.listdir('output')))

try:   # offer the download in Colab
    from google.colab import files
    files.download('output/cannibalization_report.xlsx')
except ImportError:
    pass

Saved to output/: ['cannibalization_report.csv', 'cannibalization_report.xlsx', 'daily_leader.csv', 'page_pairs.csv', 'query_page_detail.csv']


## Limits and honest caveats

* **Search Console data is sampled and filtered.** The API returns the top rows, not every row, and leaves out anonymised (rare) queries. Long-tail cannibalisation can be missing, and very large sites hit `MAX_ROWS` sooner with the date dimension on. Narrow with `PAGE_FILTER_REGEX` or shorter date ranges if you need more depth.
* **Average position is an average.** A page at "8.0" may have ranked 3 on one search and 13 on another. Rotation uses daily averages, which is coarse but good enough to spot pages taking turns.
* **Page types come from URL patterns.** If your URLs don't reveal the page type, the "different page types" check can't help. Add a column from your CMS or crawl instead.
* **The thresholds are starting points.** On a small site, lower `MIN_QUERY_IMPRESSIONS`; on a big one, raise it.
* **Check the SERP before merging.** If Google shows guides and product pages together for a query, two of your pages ranking is often the right outcome.

## What to do next

* For each **consolidate** row, compare the two pages, merge the best content into `keep_page`, 301-redirect the other and update internal links. Re-run the notebook 4 to 8 weeks later to check the split has gone.
* For pages that compete because of their **titles and headings**, the free [On-page Duplication Analysis (Google Sheets Template)](https://www.mlforseo.com/resource/on-page-duplication-analysis-google-sheets-template/) finds duplicate and near-duplicate titles, H1s and meta descriptions.
* The companion [intent mismatch audit](https://github.com/lazarinastoy/mlforseo-code/tree/main/intent-mismatch-audit) checks whether each page attracts the intent it was built for, which is the other half of the "different page types" question. It goes with my guide [How to Build a Search Intent-First Website Architecture](https://www.mlforseo.com/machine-learning-implementation-guides/onpage-seo/search-intent-first-website-architecture/).
* To go further with intent classification and keyword clustering, see the [Semantic ML-enabled Keyword Research course](https://www.mlforseo.com/courses/semantic-ml-enabled-keyword-research/).